In [1]:
import boto3
import pandas as pd
from io import StringIO
import os
from dotenv import load_dotenv
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, TextOverviewPreset
from evidently.ui.workspace import Workspace

load_dotenv()

True

In [2]:
# Load data from S3
bucket_name = 'resume-matcher-bucket-sahil' # <-- Change to your S3 bucket name
resume_key = 'raw-data/Resume.csv'
job_desc_key = 'raw-data/job_title_des.csv'

s3 = boto3.client(
    's3',
    aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY'),
)

resume_obj = s3.get_object(Bucket=bucket_name, Key=resume_key)
df_resumes = pd.read_csv(StringIO(resume_obj['Body'].read().decode('utf-8')))

job_desc_obj = s3.get_object(Bucket=bucket_name, Key=job_desc_key)
df_job_description = pd.read_csv(StringIO(job_desc_obj['Body'].read().decode('utf-8')))

print(f"Resumes dataset shape: {df_resumes.shape}")
print(f"Job Descriptions dataset shape: {df_job_description.shape}")

Resumes dataset shape: (2484, 4)
Job Descriptions dataset shape: (2277, 3)


In [3]:
# Split resume data into reference (training) and current (production) sets
split_index_resumes = int(len(df_resumes) * 0.8)
reference_resumes = df_resumes.iloc[:split_index_resumes].copy()
current_resumes = df_resumes.iloc[split_index_resumes:].copy()

print(f"Reference resumes: {reference_resumes.shape}")
print(f"Current resumes: {current_resumes.shape}")

# For this example, we'll monitor the 'Resume_str' column
# In a real scenario, you'd also monitor drift in job descriptions

Reference resumes: (1987, 4)
Current resumes: (497, 4)


In [4]:
resume_drift_report = Report(metrics=[
    DataDriftPreset(),
    TextOverviewPreset(column_name='Resume_str')
])

# Run the report
resume_drift_report.run(reference_data=reference_resumes, current_data=current_resumes)

print("Resume data drift report generated successfully!")

Resume data drift report generated successfully!


In [ ]:
# Save report as HTML
report_path = "../monitoring/evidently/reports/resume_drift_report.html"
os.makedirs(os.path.dirname(report_path), exist_ok=True)
resume_drift_report.save_html(report_path)
print(f"Report saved to: {report_path}")

# Display report inline (if in Jupyter)
resume_drift_report

Report saved to: ../monitoring/evidently/reports/resume_drift_report.html


In [ ]:
import os
from evidently.ui.workspace import Workspace
# Create Evidently Workspace for monitoring dashboard
workspace_path = "../monitoring/evidently/workspace"
os.makedirs(workspace_path, exist_ok=True)

ws = Workspace.create(workspace_path)

# Create a project
project = ws.create_project("Resume Matching Monitoring")
project.description = "Data drift monitoring for Resumes and Job Descriptions"
project.save()

# Add report to project
ws.add_report(project.id, resume_drift_report)

print(f"\nEvidently workspace created at: {workspace_path}")
print("\nTo view the dashboard, run:")
print(f"  evidently ui --workspace {workspace_path} --port 7000")


Evidently workspace created at: ../monitoring/evidently/workspace

To view the dashboard, run:
  evidently ui --workspace ../monitoring/evidently/workspace --port 7000
